# പാഠം 13 - കോഗ്നി നോളേജ് ഗ്രാഫ്സോടെ ഏജന്റ് മെമ്മറി


## ക്രമീകരിക്കല്‍

ഈ നോട്ട്‌ബുക്ക് [**Cognee**](https://www.cognee.ai/) നോളജ് ഗ്രാഫുകളും **Microsoft Agent Framework** (MAF) ഉപയോഗിച്ച് നിലനിൽക്കുന്ന മെമ്മറി കേന്ദ്രീകരിച്ച് തന്ത്രസംരക്ഷിത **കോടിങ്ങ് അസിസ്റ്റന്റ്** എങ്ങനെ നിർമ്മിക്കാമെന്ന് കാണിക്കുന്നു.

Cognee ഘടകവുമില്ലാത്ത ടെക്സ്റ്റ് ഘടനാപരവും, തിരയാവുന്നാവും ആയ നോളജ് ഗ്രാഫായി മാറ്റുന്നു, വക്‌ടർ എംബെഡ്ഡിങ്ങ് പിന്തുണയോടെ — നിങ്ങളുടെ ഏജന്റിന് സമൃദ്ധമായ, ബന്ധം അറിയുന്ന ദീര്‍ഘകാല മെമ്മറി നൽകുന്നു.

### നിങ്ങൾ പഠിക്കാനിരിക്കുന്നു
1. **നോളജ് ഗ്രാഫുകൾ നിർമ്മിക്കുക**: ഡെവലപ്പർ പ്രൊഫൈലുകളും മികച്ച പ്രാക്ടീസുകളും ഘടനാപരമായ, തിരയാവുന്ന നോലജ് ആയി മാറ്റുക.
2. **Cognee-യെ MAF-നൊപ്പം സംയോജിപ്പിക്കുക**: `@tool` ഫംഗ്ഷനുകൾ ഉപയോഗിച്ച് MAF ഏജൻറിന് Cognee നോളജ് ഗ്രാഫ് ചോദിക്കാനിടയാക്കുക.
3. **സഷന്‍: അനുസ്മരണയുള്ള സംഭാഷണങ്ങൾ**: ഒരേ സഷനിലെ വ്യത്യസ്ത ചോദ്യങ്ങൾക്ക് പ്രസ്തുത പശ്ചാത്തലം നിലനിർത്തുക.
4. **ദീർഘകാല മെമ്മറി**: സഷനുകൾക്കുപുറത്തും പ്രധാനപ്പെട്ട നോലജ് നിലനിർത്തി പുതിയ സംഭാഷണങ്ങളിൽ തിരികെ ലഭ്യമാക്കുക.

### മുമ്പ് നിർബന്ധമായ കാര്യങ്ങൾ
- Python 3.9+
- ലോക്കലിൽ Redis ഓടുന്നു (`docker run -d -p 6379:6379 redis`) സഷന്‍ മാനേജ്മെന്റിനായി
- LLM API കീ (ഉദാ: OpenAI) — `.env`-ൽ `LLM_API_KEY` സെറ്റ് ചെയ്യുക
- `.env`-ൽ `CACHING=true` (Cognee സഷനുകൾക്കായി ആവശ്യമാണ്)
- Azure AI Foundry പ്രോജക്റ്റ് ചാറ്റ് മോഡൽ വിന്യസിച്ചിരിക്കുന്നത്
- `.env`-ല്‍ `AZURE_AI_PROJECT_ENDPOINT` եւ `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- Azure CLI-യിൽ പ്രവേശനം ലഭിച്ചിരിക്കണം (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ഏജന്റ് മെമ്മറിയുടെ തരംകൾ

ഈ നോട്ട്‌ബുക്ക് മെയിൻ ലഹരി 13 നോട്ട്‌ബുക്കിൽ നിന്നുള്ള അതേ മൂന്നു മെമ്മറി തരംകൾ പരിശോധിക്കുന്നു, പക്ഷേ ദീർഘകാല മെമ്മറി ബാക്ക്‌എൻഡ് ആയി Cognee ഉപയോഗിക്കുന്നു:

| മെമ്മറി തരം | സംവിധാനമാറ്റം | ആയുസ്സ് |
|---|---|---|
| **വർക്ക് ചെയ്യുന്നത്** | `agent.create_session()` (MAF) | ഒറ്റ സംഭാഷണം |
| **ഷോർട്ട്-ടേം** | Cognee സെഷൻ കാഷെ (Redis) | ഒറ്റ സെഷൻ |
| **ദീർഘകാല** | Cognee നോളജ് ഗ്രാഫും വക്ടറുകളും | സ്ഥിരമായത് |

### Cognee ന്‍റെ മെമ്മറി ആർക്കിടെക്ചർ
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## കോഗ്‍നീ സ്റ്റോറേജ് തയ്യാറാക്കുക


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## ഭാഗം 1 — നോളജ്ബേസ് സൃഷ്ടിക്കൽ

നമ്മുടെ കോഡിംഗ് അസിസ്റ്റന്റിനായി സമഗ്രമായ നോളജ്ബേസ് സൃഷ്ടിക്കാൻ ഞങ്ങൾ മൂന്ന് തരത്തിലുള്ള ഡാറ്റ ഉൾപ്പെടുത്തുന്നു:

1. **ഡെവലപ്പർ പ്രൊഫൈൽ** — വ്യക്തിഗത പരിചയം आणि സാങ്കേതിക പശ്ചാത്തലം
2. **Python മികച്ച പ്രവർത്തന രീതികൾ** — പ്രായോഗിക മാർഗനിർദ്ദേശങ്ങളോടെ Python ന്റെ സെൻ
3. **ചരിത്രപരമായ സംഭാഷണങ്ങൾ** — ഡെവലപ്പർമാരും AI അസിസ്റ്റന്റുകളും തമ്മിലുള്ള മുമ്പുള്ള ചോദ്യോത്തര സെഷനുകൾ


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## ജ്ഞാന ഗ്രാഫ് ദൃശ്യീകരിക്കുക

Cognee ഇത് മുതലെടുത്ത ഘടകങ്ങളും ബന്ധങ്ങളുമായി ഒരു ഇന്ററാക്ടീവ് HTML ദൃശ്യം കാണിക്കാനാകുന്നു.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify ഉപയോഗിച്ച് മെമ്മറി സമ്പന്നമാക്കുക

`memify()` നോളജ് ഗ്രാഫ് വിശകലനം ചെയ്ത് ബുദ്ധിമുട്ടുള്ള നിബന്ധനകൾ സൃഷ്ടിക്കുന്നു — ശൈലികൾ, മികച്ച ഉപായങ്ങൾ, സംജ്ഞകളിടയിൽ ബന്ധങ്ങൾ തിരിച്ചറിയുന്നു.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Part 2 — Cognee ഉപകരണങ്ങളുള്ള MAF ഏജന്റ്

ഇപ്പോൾ ഞങ്ങൾ ഒരു MAF ഏജന്റ് സൃഷ്ടിക്കുന്നു, അത് `@tool` ഫംഗ്ഷനുകൾ വഴി Cognee നു സ്വന്തമായ നോളഡ്ജ് ഗ്രാഫിനെ ചോദ്യം ചെയ്യാൻ കഴിയും. ഇതിലൂടെ ഏജന്റ് സെഷനുകളുടെ സംഭാഷണസന്ദർഭം നിലനിർത്തിക്കൊണ്ടിരിക്കുമ്പോൾ ഗ്രാഫ്-അറിയുന്ന സെമാന്റിക് തിരച്ചിലിന്റെ മുഴുവൻ ശക്തിയും പ്രയോജനപ്പെടുത്താൻ കഴിയും.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## സെഷനുകളോടുള്ള വർക്കിംഗ് മെമ്മറി

`AgentSession` (`agent.create_session()` വഴി സൃഷ്ടിക്കപ്പെടുന്നു) സെഷനിനുള്ളിലെ വർക്കിംഗ് മെമ്മറിയു നൽകുന്നു. ഏജന്റ് മുമ്പത്തെ സന്ദേശങ്ങളെ തിരിച്ചു കാണാനാകും കൂടാതെ Cogneeയുടെ ദീർഘകാല ജ്ഞാന ഗ്രാഫ്에도 ചോദ്യം ചെയ്യാൻ കഴിയും.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## New Session — Long-Term Memory Persists

പുതിയ സെഷൻ തുടങ്ങുമ്പോൾ വർക്കിംഗ് മെമ്മറി ക്ലിയർ ചെയ്യപ്പെടും, പക്ഷേ Cognee നോളജ് ഗ്രാഫ് ഇപ്പോഴും ലഭ്യമാണ്. ഏജന്റ് തികച്ചും പുതിയ സംഭാഷണത്തിൽ അതേ ദീര്‍ഘകാല അറിവ് തിരികെ ലഭ്യമാക്കാനാകും.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## സംക്ഷേപം

ഈ നോട്ട്ബുക്കിൽ നിങ്ങൾ നിർമ്മിച്ചത് **MAF-യുടെ വര്‍ക്കിംഗ് മെമ്മറി** (`agent.create_session()`) **Cogneeയുടെ ദീര്‍ഘകാല അറിവുകളുടെ ഗ്രാഫുമായി സംയോജിപ്പിക്കുന്ന** ഒരു കോഡിംഗ് അസിസ്റ്റന്റ് ആണ്.

### നിങ്ങൾ പഠിച്ചത്
1. **അറിവുകളുടെ ഗ്രാഫ് നിർമ്മാവ്**: Cognee ഘടിതരീതിയിലുള്ള ടെക്സ്റ്റ് സ്വീകരിച്ച് ഒരു ഗ്രാഫും വെക്ടർ മെമ്മറിയും തയാറാക്കുന്നു.
2. **memify ഉപയോഗിച്ച് ഗ്രാഫ് സമൃദ്ധീകരണം**: നിലനില്ക്കുന്ന ഗ്രാഫ് അടിസ്ഥാനമാക്കി പ്രാപ്തമായ വസ്തുതകളും സമൃദ്ധമായ ബന്ധങ്ങളും സൃഷ്ടിക്കുന്നു.
3. **MAF + Cognee സംയോജനം**: `@tool` ഫങ്ഷനുകൾ MAF ഏജന്റുകൾക്ക് Cogneeയുടെ ഗ്രാഫിനെ സ്വാഭാവികമായി ക്വറി ചെയാനും സഹായിക്കുന്നു.
4. **വര്‍ക്കിംഗ് മെമ്മറി + ദീര്‍ഘകാല മെമ്മറി**: `AgentSession` (`agent.create_session()` വഴി) സെഷന്‍ സന്ദർഭം നൽകുന്നതും Cognee സ്ഥിരമായ അറിവ് നൽകുന്നതുമാണ്.
5. **NodeSets ഉപയോഗിച്ച് ഫിൽട്ടർ ചെയ്ത തിരച്ചിൽ**: അറിവിന്റെ ഗ്രാഫിൽ നിന്നും പ്രത്യേക ഉപസമൂഹങ്ങളെ ലക്ഷ്യമിട്ട് തിരയുന്നു (ഉദാഹരണം: പ്രിൻസിപ്പിൾസ് മാത്രം).

### പ്രധാന തിയറികൾ
- **Cognee** എളുപ്പത്തില്‍ ടെക്സ്റ്റ് ഘടിത ആകൃതിയിലുള്ള, ബന്ധ-അറിയുന്ന മെമ്മറിയായി മാറ്റുന്നു — സാധാരണ വെക്ടർ സ്റ്റോറിനേക്കാൾ ശക്തമാണ്.
- **`@tool` ഫങ്ഷനുകൾ** MAF ഏജന്റുകൾക്കും ബാഹ്യ അറിവു സംവിധാനങ്ങൾക്കും ഇടയിൽ നേർക്കാഴ്ച പാലിക്കുന്നു.
- **`AgentSession`** (`agent.create_session()` വഴി) ഓരോ സംഭാഷണത്തിന്റെയും സന്ദർഭം ദീർഘകാല അറിവിൽ നിന്നും വേര്‍തിരിക്കുന്നു.
- ഒരേ അറിവുകളുടെ ഗ്രാഫ് പല സെഷനുകൾക്കും ഏജന്റുകൾക്കും സേവനം നൽകുന്നു.

### യഥാർത്ഥ പ്രയോഗങ്ങൾ
- **ഡെവലപ്പർ കോപൈലറ്റുകൾ**: കോഡ് റിവ്യു, സംഭവ വിശകലനം, ആർക്കിടെക്ചർ അസിസ്റ്റന്റുകൾ
- **ഉപഭോക്തൃ-സാമ്ദ്രമായ കോപൈലറ്റുകൾ**: ഉൽപ്പന്ന ഡോക്യുമെന്റുകൾ, FAQ-കൾ, CRM നോട്ടുകൾ എന്നിവിടങ്ങളിൽ പിന്തുണ ഏജന്റുകൾ
- **അകത്ത് വിദഗ്ദ്ധ കോപൈലറ്റുകൾ**: നയം, നിയമം, സുരക്ഷ സംബന്ധിച്ച ഗൈഡ്‌ലൈൻസ് പരിശോധിക്കുന്ന അസിസ്റ്റന്റുകൾ
- **ഏകജാലക ഡാറ്റാ ലെയറുകൾ**: ഘടിതവും ഘടിതരീതിയല്ലാത്തതുമായ ഡാറ്റകൾ ഒരെേടായി സംയോജിപ്പിച്ച് ക്വറി ചെയ്യാവുന്ന ഒരു ഗ്രാഫ് സൃഷ്ടിക്കുക

### അടുത്ത പടികള്‍
- Cognee-യിൽ ടൈംപറൽ അവെയർനെസ് പരീക്ഷിക്കുക
- ഡൊമെയിന്-സ്പെസിഫിക് ഗ്രാഫ് ഗുണമേന്മക്ക് OWL ഓണ്ടോളജി നിർവചിക്കുക
- ലഭ്യത മെച്ചപ്പെടുത്താൻ ഉപയോക്തൃ പ്രതികരണ ചക്രങ്ങൾ ചേർക്കുക
- ഒരേ Cognee മെമ്മറി ലെയർ പങ്കിടുന്ന ബഹുഏജന്റ് സംവിധാനങ്ങൾക്കായി സ്കെയിൽ ചെയ്യുക


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അസ്വീകാരം**:
ഈ രേഖ [Co-op Translator](https://github.com/Azure/co-op-translator) എന്ന എഐ പരിഭാഷ സേവനം ഉപയോഗിച്ച് പരിഭാഷ ചെയ്തതാണ്. നാം കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, സ്വയം പ്രവർത്തിക്കുന്ന പരിഭാഷകളിൽ പിഴവുകളോ അശുദ്ധികളോ ഉണ്ടായിരിക്കാമെന്നുള്ള കാര്യം ദയവായി മനസ്സിലാക്കുക. സാദ്ധ്യമായ എല്ലാ കാര്യങ്ങൾക്കും അവയുടെ മാതൃഭാഷയിൽ ഉള്ള മാതൃപ്രമാണം പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കണം. നിർണായകമായ വിവരങ്ങൾക്കായി പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്തു. ഈ പരിഭാഷ ഉപയോഗിക്കുന്നതിൽ നിന്നുണ്ടാകാവുന്ന എന്തെങ്കിലും തെറ്റിദ്ധാരണകൾക്കും ഓഫ് വ്യാഖ്യാനങ്ങൾക്കും ഞങ്ങൾ ഉത്തരവാദിപ്പെടുന്നില്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
